In [1]:
import sys
import os

print(os.getcwd())

# 如果当前工作目录就是模块所在目录，则无需添加；否则手动添加
if r'd:\VSCODEProject\PPO\enery' not in sys.path:
    sys.path.insert(0, r'd:\VSCODEProject\PPO\enery')
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

d:\VSCODEProject\PPO\enery


In [ ]:
import numpy as np
from BossDuelEnv import BossDuelEnv

def get_boss_action_name(state):
    """
    从状态中提取 Boss 当前动作名称
    
    Args:
        state (numpy.ndarray): 包含 Boss 动作状态的状态数组，前三个元素分别对应重击、轻击、蓄力动作的激活状态
    
    Returns:
        str: Boss 当前动作名称，可能的值为 "重击"、"轻击" 或 "蓄力"
    """
    """从状态中提取 Boss 当前动作名称"""
    boss_action = np.argmax(state[:3])
    return ["重击", "轻击", "蓄力"][boss_action]

def print_state(env, state, turn):
    """打印当前回合开始时的状态（Boss 即将使用的动作）"""
    print("\n" + "="*60)
    print(f"【第 {turn} 回合】")
    print(f"Agent HP: {env.agent_health:.1f} / {env.max_health}")
    print(f"Boss  HP: {env.boss_health:.1f} / {env.boss_health_max}")
    print(f"能量: {env.agent_energy:.1f} / {env.max_energy}")
    print(f"体力: {env.agent_stamina:.1f} / {env.max_stamina}")
    boss_action = get_boss_action_name(state)
    print(f"Boss 即将使用的动作: {boss_action}", end="")
    if boss_action == "蓄力":
        print("  ⚠️ 蓄力后下一回合攻击暴击！可用大招打断！")
    else:
        print()
    if env.boss_charge:
        print("🔥 Boss 当前处于蓄力状态！下回合攻击必定暴击且无法格挡/闪避！🔥")
    print("="*60)

def main():
    env = BossDuelEnv()
    done = False
    turn = 1
    state = env.reset()
    # 累计奖励变量（仅用于显示）
    total_reward = 0

    print("欢迎来到 Boss Duel！")
    print("动作指令：0=格挡, 1=闪避, 2=大招, 3=普攻")
    print("输入 q 退出游戏\n")

    while not done:
        # 显示当前状态（Boss 即将使用的动作）
        print_state(env, state, turn)

        # 获取玩家输入
        user_input = input("请选择动作 (0-3, q退出): ").strip().lower()
        if user_input == 'q':
            break
        if not user_input.isdigit():
            print("无效输入，请输入 0-3 的数字")
            continue
        action = int(user_input)
        if action not in [0, 1, 2, 3]:
            print("动作范围 0-3")
            continue

        # 执行动作
        next_state, reward, done, info = env.step(action)
        total_reward += reward

        # 显示本回合结果
        print("\n--- 本回合结果 ---")
        boss_prev = ["重击", "轻击", "蓄力"][info['boss_action']]
        agent_prev = ["格挡", "闪避", "大招", "普攻"][action]
        print(f"Boss 实际动作: {boss_prev} | 你的动作: {agent_prev}")
        print(f"造成伤害: {info['damage_dealt']} | 受到伤害: {info['damage_taken']}")
        print(f"本回合奖励: {reward:.2f}")
        print(f"累计奖励: {total_reward:.2f}")

        # 检查游戏结束
        if done:
            print("\n游戏结束！")
            if env.agent_health <= 0:
                print("你输了...")
            elif env.boss_health <= 0:
                print("你赢了！")
            else:
                print("平局")
            print(f"最终累计奖励: {total_reward:.2f}")
            break

        # 更新状态和回合数
        state = next_state
        turn += 1

if __name__ == "__main__":
    
    main()

欢迎来到 Boss Duel！
动作指令：0=格挡, 1=闪避, 2=大招, 3=普攻
输入 q 退出游戏


【第 1 回合】
Agent HP: 100.0 / 100
Boss  HP: 200.0 / 200
能量: 0.0 / 100
体力: 100.0 / 100
Boss 即将使用的动作: 轻击

--- 本回合结果 ---
Boss 实际动作: 轻击 | 你的动作: 格挡
造成伤害: 0 | 受到伤害: 2
本回合奖励: -0.40
累计奖励: -0.40

【第 2 回合】
Agent HP: 98.0 / 100
Boss  HP: 200.0 / 200
能量: 15.0 / 100
体力: 100.0 / 100
Boss 即将使用的动作: 重击

--- 本回合结果 ---
Boss 实际动作: 重击 | 你的动作: 闪避
造成伤害: 0 | 受到伤害: 0
本回合奖励: 1.00
累计奖励: 0.60

【第 3 回合】
Agent HP: 98.0 / 100
Boss  HP: 200.0 / 200
能量: 20.0 / 100
体力: 85.0 / 100
Boss 即将使用的动作: 重击

--- 本回合结果 ---
Boss 实际动作: 重击 | 你的动作: 格挡
造成伤害: 0 | 受到伤害: 5
本回合奖励: -1.00
累计奖励: -0.40

【第 4 回合】
Agent HP: 93.0 / 100
Boss  HP: 200.0 / 200
能量: 35.0 / 100
体力: 90.0 / 100
Boss 即将使用的动作: 轻击

--- 本回合结果 ---
Boss 实际动作: 轻击 | 你的动作: 格挡
造成伤害: 0 | 受到伤害: 2
本回合奖励: -0.40
累计奖励: -0.80

【第 5 回合】
Agent HP: 91.0 / 100
Boss  HP: 200.0 / 200
能量: 50.0 / 100
体力: 95.0 / 100
Boss 即将使用的动作: 轻击

--- 本回合结果 ---
Boss 实际动作: 轻击 | 你的动作: 格挡
造成伤害: 0 | 受到伤害: 2
本回合奖励: -0.40
累计奖励: -1.20

【第 6 回合】
Agent HP: 89.0 / 100

: 